In [8]:
import warnings
warnings.filterwarnings("ignore", category=Warning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

from sklearn.metrics import (
  confusion_matrix,
  ConfusionMatrixDisplay,
  classification_report,
  accuracy_score
)

wineData = load_wine()
X = pd.DataFrame(
  wineData.data,
  columns=wineData.feature_names
)

y = pd.Series(wineData.target)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

print(X.head())
print(y.value_counts().sort_index())

Shape of X: (178, 13)
Shape of y: (178,)
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_dil

In [9]:
# First: separate training data from the remaining data
X_train, X_temp, y_train, y_temp = train_test_split(
  X, y,
  test_size=0.40,
  random_state=42,
  stratify=y
)

# Second: split the remaining data into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
  X_temp, y_temp,
  test_size=0.50,
  random_state=42,
  stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (106, 13)
Validation: (36, 13)
Testing: (36, 13)


In [10]:
scaler = StandardScaler()

# Fit only on training data
X_train = scaler.fit_transform(X_train)

# Transform validation and test data using the same scaler
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert the data into PyTorch tensors:

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_val = torch.tensor(y_val.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [11]:
class RegressionNN(nn.Module):
  def __init__(self, hidden_sizes):
    super().__init__()
    layers = []
    input_size = 1 # set the input size according to your dataset
    output_size = 1 # set according to the dataset

    for hidden_size in hidden_sizes:
      layers.append(
        nn.Linear(input_size, hidden_size)
    )
    layers.append(nn.Tanh())

    input_size = hidden_size

    # One output neuron because this is regression
    layers.append(nn.Linear(input_size, output_size))

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x)

In [12]:
search_space = [
  # Architecture, learning rate, weight decay
  ([2], 0.01, 0.0),
  ([8], 0.01, 0.0),
  ([16], 0.01, 0.0),
  ([64], 0.01, 0.0),
  ([16, 16], 0.01, 0.0),
  ([64, 64], 0.01, 0.0),
  ([16], 0.001, 0.0),
  ([16], 0.01, 0.01),
]

results = []
candidate_models = []

for hidden_sizes, lr, wd in search_space:
  torch.manual_seed(42)

  model = RegressionNN(hidden_sizes)

  train_losses, val_losses = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    epochs=500,
    learning_rate=lr,
    weight_decay=wd
)

# Use the final validation loss for this simple demonstration
results.append({
  "hidden_layers": str(hidden_sizes),
  "learning_rate": lr,
  "weight_decay": wd,
  "train_mse": train_losses[-1],
  "validation_mse": val_losses[-1]
})

candidate_models.append(model)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("validation_mse")

NameError: name 'train_model' is not defined

In [ ]:
# Retrieve the best model
best_model = candidate_models[
  results_df.index[0]
]

best_model.eval()

with torch.no_grad():

  test_predictions = best_model(X_test)

  # Convert output logits into predicted class labels
  y_pred = torch.argmax(
    test_predictions,
    dim=1
  )
# Calculate test accuracy
test_accuracy = accuracy_score(
  y_test.numpy(),
  y_pred.numpy()
)

print("Test Accuracy:", test_accuracy)

# Generate the confusion matrix
cm = confusion_matrix(
  y_test.numpy(),
  y_pred.numpy(),
  labels=[0, 1, 2]
)

disp = ConfusionMatrixDisplay(
  confusion_matrix=cm,
  display_labels=wineData.target_names
)

disp.plot(
  cmap="Blues",
  values_format="d"
)

plt.title("Confusion Matrix - Test Dataset")
plt.show()

# Generate the classification report
print(
  classification_report(
    y_test.numpy(),
    y_pred.numpy(),
    target_names=wineData.target_names
  )
)